In [9]:
import numpy as np
import pandas as pd 

In [10]:
!pip install qiskit==0.39.2

  Using cached qiskit-0.39.2.tar.gz (13 kB)
  Preparing metadata (setup.py) ... done
  Using cached qiskit_terra-0.22.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.5 kB)
  Using cached qiskit-aer-0.11.1.tar.gz (6.5 MB)
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [6]:

import numpy as np
import random
import re


from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, execute, Aer


from qiskit.tools.visualization import circuit_drawer, plot_histogram
from qiskit.quantum_info import Statevector
from qiskit.visualization import array_to_latex

ModuleNotFoundError: No module named 'qiskit'

In [ ]:

qr = QuantumRegister(2, name="qr")
cr = ClassicalRegister(4, name="cr")

In [ ]:
singlet = QuantumCircuit(qr, cr, name='singlet')
singlet.x(qr[0])
singlet.x(qr[1])
singlet.h(qr[0])
singlet.cx(qr[0],qr[1])

In [ ]:

measureA1 = QuantumCircuit(qr, cr, name='measureA1')
measureA1.h(qr[0])
measureA1.measure(qr[0],cr[0])

measureA2 = QuantumCircuit(qr, cr, name='measureA2')
measureA2.s(qr[0])
measureA2.h(qr[0])
measureA2.t(qr[0])
measureA2.h(qr[0])
measureA2.measure(qr[0],cr[0])

measureA3 = QuantumCircuit(qr, cr, name='measureA3')
measureA3.measure(qr[0],cr[0])


measureB1 = QuantumCircuit(qr, cr, name='measureB1')
measureB1.s(qr[1])
measureB1.h(qr[1])
measureB1.t(qr[1])
measureB1.h(qr[1])
measureB1.measure(qr[1],cr[1])

measureB2 = QuantumCircuit(qr, cr, name='measureB2')
measureB2.measure(qr[1],cr[1])

measureB3 = QuantumCircuit(qr, cr, name='measureB3')
measureB3.s(qr[1])
measureB3.h(qr[1])
measureB3.tdg(qr[1])
measureB3.h(qr[1])
measureB3.measure(qr[1],cr[1])

aliceMeasurements = [measureA1, measureA2, measureA3]
bobMeasurements = [measureB1, measureB2, measureB3]

In [ ]:
pip install pylatexenc


In [ ]:
measureA1.draw(output = 'mpl') # X
measureA2.draw(output = 'mpl') # W
measureA3.draw(output = 'mpl') # Z
measureB1.draw(output = 'mpl') # W 
measureB2.draw(output = 'mpl') # Z
measureB3.draw(output = 'mpl') # V

In [ ]:
numberOfSinglets = 500

In [ ]:
aliceMeasurementChoices = [random.randint(1, 3) for i in range(numberOfSinglets)] 
bobMeasurementChoices = [random.randint(1, 3) for i in range(numberOfSinglets)]

In [ ]:
circuits = [] 

for i in range(numberOfSinglets):
    circuitName = str(i) + ':A' + str(aliceMeasurementChoices[i]) + '_B' + str(bobMeasurementChoices[i])
    circuitName = singlet + aliceMeasurements[aliceMeasurementChoices[i]-1] + bobMeasurements[bobMeasurementChoices[i]-1] 
    circuits.append(circuitName)

In [ ]:
print(circuits[0].name)

In [ ]:
backend=Aer.get_backend('qasm_simulator')
result = execute(circuits, backend=backend, shots=1).result()  

In [ ]:
result.get_counts(circuits[0])
plot_histogram(result.get_counts(circuits[0]))

In [ ]:
abPatterns = [
    re.compile('..00$'), 
    re.compile('..01$'),
    re.compile('..10$'), 
    re.compile('..11$')  
]

In [ ]:
aliceResults = [] 
bobResults = [] 


for i in range(numberOfSinglets):

    res = list(result.get_counts(circuits[i]).keys())[0] 
    
    if abPatterns[0].search(res): 
        aliceResults.append(-1) 
        bobResults.append(-1) 
    if abPatterns[1].search(res):
        aliceResults.append(1)
        bobResults.append(-1)
    if abPatterns[2].search(res):
        aliceResults.append(-1) 
        bobResults.append(1) 
    if abPatterns[3].search(res): 
        aliceResults.append(1)
        bobResults.append(1)

In [ ]:
aliceKey = [] # Alice's key string k
bobKey = [] # Bob's key string k'


for i in range(numberOfSinglets):
    if (aliceMeasurementChoices[i] == 2 and bobMeasurementChoices[i] == 1) or (aliceMeasurementChoices[i] == 3 and bobMeasurementChoices[i] == 2):
        aliceKey.append(aliceResults[i]) 
        bobKey.append(- bobResults[i]) 
        
keyLength = len(aliceKey)

In [ ]:
abKeyMismatches = 0


for j in range(keyLength):
    if aliceKey[j] != bobKey[j]:
        abKeyMismatches += 1

In [ ]:
print("alice key: ", aliceKey)

print("bob key : ", bobKey)
print(type(aliceKey))

In [ ]:
def convert_list_to_string(bit_list):
    """
    Convert a list of -1 and 1 (integers) into a binary string by replacing -1 with '0' and 1 with '1'.
    
    :param bit_list: List of integers (-1 or 1).
    :return: A string representing the binary values.
    """
    if not isinstance(bit_list, list):
        raise TypeError("Input must be a list.")

    # Convert all elements to integers (in case they are mistakenly strings)
    try:
        bit_list = [int(bit) for bit in bit_list]
    except ValueError:
        raise ValueError("All elements in the list must be either -1 or 1.")

    if not all(bit in [-1, 1] for bit in bit_list):
        raise ValueError("The list must contain only -1 or 1 as integers.")

    return ''.join('0' if bit == -1 else '1' for bit in bit_list)

aliceKey = convert_list_to_string(aliceKey)
bobKey = convert_list_to_string(bobKey)

print("alice key 500: ", aliceKey)


print("bob key 500: ", bobKey)
print(aliceKey == bobKey)

In [ ]:
import binascii

def binary_to_bytes(binary_str: str) -> bytes:
    """Convert a binary string to bytes."""
    # Pad the binary string with leading zeros to make it a multiple of 8
    padded_binary_str = binary_str.zfill((len(binary_str) + 7) // 8 * 8)
    
    # Convert binary string to bytes
    byte_data = int(padded_binary_str, 2).to_bytes(len(padded_binary_str) // 8, byteorder='big')
    
    return byte_data



In [ ]:
aliceKey = binary_to_bytes(aliceKey)
bobKey = binary_to_bytes(bobKey)
print(aliceKey)
print(bobKey)
print(aliceKey == bobKey)

In [ ]:
def save_bytes_to_bin_file(byte_data, filename):
    """
    Save bytes to a binary file.

    :param byte_data: Bytes-like object to write to the file.
    :param filename: The name of the binary file to save.
    """
    with open(filename, "wb") as bin_file:
        bin_file.write(byte_data)

In [ ]:
save_bytes_to_bin_file(aliceKey, 'aliceKey.bin')
save_bytes_to_bin_file(bobKey, 'bobKey.bin')

In [ ]:
from IPython.display import FileLink

# Display a download link
FileLink("aliceKey.bin")

In [ ]:
FileLink("bobKey.bin")


In [ ]:

def chsh_corr(result):
    
    countA1B1 = [0, 0, 0, 0] # XW observable
    countA1B3 = [0, 0, 0, 0] # XV observable
    countA3B1 = [0, 0, 0, 0] # ZW observable
    countA3B3 = [0, 0, 0, 0] # ZV observable

    for i in range(numberOfSinglets):

        res = list(result.get_counts(circuits[i]).keys())[0]
        if (aliceMeasurementChoices[i] == 1 and bobMeasurementChoices[i] == 1):
            for j in range(4):
                if abPatterns[j].search(res):
                    countA1B1[j] += 1

        if (aliceMeasurementChoices[i] == 1 and bobMeasurementChoices[i] == 3):
            for j in range(4):
                if abPatterns[j].search(res):
                    countA1B3[j] += 1

        if (aliceMeasurementChoices[i] == 3 and bobMeasurementChoices[i] == 1):
            for j in range(4):
                if abPatterns[j].search(res):
                    countA3B1[j] += 1
                    
        if (aliceMeasurementChoices[i] == 3 and bobMeasurementChoices[i] == 3):
            for j in range(4):
                if abPatterns[j].search(res):
                    countA3B3[j] += 1
                    
    total11 = sum(countA1B1)
    total13 = sum(countA1B3)
    total31 = sum(countA3B1)
    total33 = sum(countA3B3)      
                    
    expect11 = (countA1B1[0] - countA1B1[1] - countA1B1[2] + countA1B1[3])/total11 # -1/sqrt(2)
    expect13 = (countA1B3[0] - countA1B3[1] - countA1B3[2] + countA1B3[3])/total13 # 1/sqrt(2)
    expect31 = (countA3B1[0] - countA3B1[1] - countA3B1[2] + countA3B1[3])/total31 # -1/sqrt(2)
    expect33 = (countA3B3[0] - countA3B3[1] - countA3B3[2] + countA3B3[3])/total33 # -1/sqrt(2) 
    
    corr = expect11 - expect13 + expect31 + expect33 # calculate the CHSC correlation value (3)
    
    return corr

In [ ]:
corr = chsh_corr(result) 


print('CHSH correlation value: ' + str(round(corr, 3)))


print('Length of the key: ' + str(keyLength))
print('Number of mismatching bits: ' + str(abKeyMismatches) + '\n')

In [ ]:

measureEA2 = QuantumCircuit(qr, cr, name='measureEA2')
measureEA2.s(qr[0])
measureEA2.h(qr[0])
measureEA2.t(qr[0])
measureEA2.h(qr[0])
measureEA2.measure(qr[0],cr[2])

measureEA3 = QuantumCircuit(qr, cr, name='measureEA3')
measureEA3.measure(qr[0],cr[2])

measureEB1 = QuantumCircuit(qr, cr, name='measureEB1')
measureEB1.s(qr[1])
measureEB1.h(qr[1])
measureEB1.t(qr[1])
measureEB1.h(qr[1])
measureEB1.measure(qr[1],cr[3])

measureEB2 = QuantumCircuit(qr, cr, name='measureEB2')
measureEB2.measure(qr[1],cr[3])

eveMeasurements = [measureEA2, measureEA3, measureEB1, measureEB2]

In [ ]:

eveMeasurementChoices = []

for j in range(numberOfSinglets):      
    if random.uniform(0, 1) <= 0.5: 
        eveMeasurementChoices.append([0, 2])
    else: 
        eveMeasurementChoices.append([1, 3])

In [ ]:
circuits = [] 

for j in range(numberOfSinglets):
    
    circuitName = str(j) + ':A' + str(aliceMeasurementChoices[j]) + '_B' + str(bobMeasurementChoices[j] + 2) + '_E' + str(eveMeasurementChoices[j][0]) + str(eveMeasurementChoices[j][1] - 1)
    
    circuitName = singlet + eveMeasurements[eveMeasurementChoices[j][0]-1] + eveMeasurements[eveMeasurementChoices[j][1]-1] + aliceMeasurements[aliceMeasurementChoices[j]-1] +  bobMeasurements[bobMeasurementChoices[j]-1]
    
    circuits.append(circuitName)

In [ ]:
backend=Aer.get_backend('qasm_simulator')
result = execute(circuits, backend=backend, shots=1).result()
print(result)

In [ ]:
print(str(circuits[0].name) + '\t' + str(result.get_counts(circuits[0])))

In [ ]:
plot_histogram(result.get_counts(circuits[0]))

In [ ]:
ePatterns = [
    re.compile('00..$'), 
    re.compile('01..$'), 
    re.compile('10..$'),
    re.compile('11..$')  
]

In [ ]:
aliceResults = [] 
bobResults = [] 

eveResults = [] 

for j in range(numberOfSinglets):
    
    res = list(result.get_counts(circuits[j]).keys())[0] 
    
    # Alice and Bob
    if abPatterns[0].search(res): 
        aliceResults.append(-1) 
        bobResults.append(-1) 
    if abPatterns[1].search(res):
        aliceResults.append(1)
        bobResults.append(-1)
    if abPatterns[2].search(res): 
        aliceResults.append(-1)
        bobResults.append(1) 
    if abPatterns[3].search(res): 
        aliceResults.append(1)
        bobResults.append(1)

    # Eve
    if ePatterns[0].search(res): 
        eveResults.append([-1, -1]) 
    if ePatterns[1].search(res):
        eveResults.append([1, -1])
    if ePatterns[2].search(res):
        eveResults.append([-1, 1])
    if ePatterns[3].search(res):
        eveResults.append([1, 1])

In [ ]:
aliceKey = []
bobKey = []
eveKeys = [] 


for j in range(numberOfSinglets):
    
    if (aliceMeasurementChoices[j] == 2 and bobMeasurementChoices[j] == 1) or (aliceMeasurementChoices[j] == 3 and bobMeasurementChoices[j] == 2):  
        aliceKey.append(aliceResults[j]) 
        bobKey.append(-bobResults[j]) 
        eveKeys.append([eveResults[j][0], -eveResults[j][1]])

keyLength = len(aliceKey) 

In [ ]:
abKeyMismatches = 0
eaKeyMismatches = 0 
ebKeyMismatches = 0 

for j in range(keyLength):
    if aliceKey[j] != bobKey[j]: 
        abKeyMismatches += 1
    if eveKeys[j][0] != aliceKey[j]:
        eaKeyMismatches += 1
    if eveKeys[j][1] != bobKey[j]:
        ebKeyMismatches += 1

In [ ]:
eaKnowledge = (keyLength - eaKeyMismatches)/keyLength 
ebKnowledge = (keyLength - ebKeyMismatches)/keyLength 

In [ ]:
corr = chsh_corr(result)

In [ ]:
# CHSH inequality test
print('CHSH correlation value: ' + str(round(corr, 3)) + '\n')

# Keys
print('Length of the key: ' + str(keyLength))
print('Number of mismatching bits: ' + str(abKeyMismatches) + '\n')

print('Eve\'s knowledge of Alice\'s key: ' + str(round(eaKnowledge * 100, 2)) + ' %')
print('Eve\'s knowledge of Bob\'s key: ' + str(round(ebKnowledge * 100, 2)) + ' %')